# 03 · Sepsis LSTM

> **Synthetic data only.** Every dataset used in this notebook is synthetic (Synthea, seed 42) or research-use-only (NIH ChestX-ray14). There is no PHI. None of these models is clinically validated or cleared for patient care.

Walk through the sepsis early-warning LSTM defined in `medflow_ml.models.lstm`. We inspect the architecture/hyper-parameters, build a tiny synthetic batch, and show a minimal training-loop snippet. Cells are illustrative and need not execute.

## Inspect the model config
The `LSTMConfig` dataclass imports without torch, so we can read the hyper-parameters cheaply.

In [ ]:
from __future__ import annotations
from medflow_ml.models.lstm import LSTMConfig, build_module
from medflow_ml.features.vitals import SEQUENCE_STEPS, VITALS_FEATURES

cfg = LSTMConfig()
print('input_size :', cfg.input_size, '== len(VITALS_FEATURES) ==', len(VITALS_FEATURES))
print('hidden/layers/dropout:', cfg.hidden_size, cfg.num_layers, cfg.dropout)
print('sequence length:', SEQUENCE_STEPS)

## Build the LightningModule
`build_module` imports torch lazily. In a GPU-less notebook this still constructs a CPU module.

In [ ]:
model = build_module(cfg)
print(type(model).__name__)
n_params = sum(p.numel() for p in model.parameters())
print('trainable params:', n_params)

## A synthetic batch
Shape `[batch, 24, 5]` — the normalized vitals window. Labels are per-window 0/1 sepsis flags.

In [ ]:
import torch
torch.manual_seed(42)
batch_size = 8
x = torch.randn(batch_size, SEQUENCE_STEPS, len(VITALS_FEATURES))
y = (torch.rand(batch_size) > 0.7).float()
logits = model(x)
print('logits shape:', tuple(logits.shape))
print('probs        :', torch.sigmoid(logits).detach().round(decimals=3).tolist())

## Minimal training-loop snippet
The real job uses PyTorch Lightning + MLflow with a time-aware split. This bare loop shows the mechanics.

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
loss_fn = torch.nn.BCEWithLogitsLoss()
for step in range(3):  # tiny demo, not convergence
    opt.zero_grad()
    loss = loss_fn(model(x), y)
    loss.backward(); opt.step()
    print(f'step {step}: loss={loss.item():.4f}')

### Notes
* Production training: `python -m medflow_ml.jobs.train_sepsis` logs AUROC/AUPRC and registers `sepsis-ews`.
* The time-aware split prevents future-into-past leakage.
* Next: a tabular model (notebook 04).